# 직접 해보기 07 · 펄서 후보 분류기 시험하기

파크스 망원경의 실제 펄서 후보 17,898행(HTRU2, UCI doi:10.24432/C5DK6R, CC BY 4.0)으로 분류기를 시험한다. 같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
X, y = A.load_raw()
split = A.load_split()
model = __import__('json').load(open('data/model.json', encoding='utf-8'))
print(len(y), '행 · 펄서', sum(y), '· 지문', A.sha8('data/HTRU_2.csv'))
print({n: split.count(n) for n in ('train', 'validation', 'test')})

## 1. 점수를 다시 계산한다
`model.json`의 평균·표준편차·가중치로 17,898행의 점수를 만든다. (처음부터 학습하려면 `A.retrain()`, NumPy 필요)

In [ ]:
s = A.scores_from_model(X, model)
print(min(s), max(s))

## 2. 기준선: 모두 ‘아님’이라고 답하면
시험 묶음 정확도와 재현율을 먼저 적어 둔다.

In [ ]:
te = [i for i, p in enumerate(split) if p == 'test']
npos = sum(y[i] for i in te)
print('시험', len(te), '행 · 펄서', npos, '· 모두 아님 정확도', round((len(te) - npos) / len(te), 3), '· 재현율 0')

## 3. 검증 묶음에서 문턱을 고른다
재현율 90% 이상을 남기는 가장 높은 문턱(0.01 간격).

In [ ]:
t = A.pick_threshold(y, s, split, 0.90)
print('문턱', t, A.confusion(y, s, split, 'validation', t))

## 4. 시험 묶음을 한 번 연다
혼동행렬의 네 칸, 재현율, 정밀도. 이 뒤에 바꾼 문턱은 ‘사후 탐색’이다.

In [ ]:
r = A.confusion(y, s, split, 'test', t)
print(r)
print('문턱 0.5(사후 탐색)', A.confusion(y, s, split, 'test', 0.5))

## 5. 점수 분포와 문턱을 그린다
matplotlib이 없으면 `pip install matplotlib`.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
for lab, c, col in ((1, 'pulsar', '#5C5FB0'), (0, 'not pulsar', '#8a867d')):
    ax.hist([s[i] for i in te if y[i] == lab], bins=50, range=(0, 1), alpha=.85, color=col, label=c)
ax.set_yscale('log'); ax.axvline(t, color='#D9733D'); ax.set_xlabel('score'); ax.set_ylabel('rows (log)'); ax.legend()
plt.show()

## 6. 문턱을 훑어 본다
문턱에 따라 남긴 펄서·놓친 펄서·잘못 남긴 것이 어떻게 바뀌는가? 어느 문턱이 ‘좋은’ 문턱인지는 무엇을 더 비싸게 여기느냐에 달렸다.

In [ ]:
print('문턱  남김/놓침/잘못남김  재현율  정밀도')
for k in (5, 10, 16, 25, 40, 50, 70, 90):
    c = A.confusion(y, s, split, 'test', k / 100)
    print(f"{k/100:.2f}  {c['tp']:>3}/{c['fn']:>3}/{c['fp']:>3}   {c['recall']:.3f}  {c['precision'] or 0:.3f}")

## 7. 기록
고른 문턱과 이유, 시험을 연 뒤 한 일을 구분해 적는다. 놓친 펄서 42개의 여덟 숫자를 표로 저장해 남긴 펄서와 비교해 보는 것이 다음 걸음이다.

In [ ]:
missed = [i for i in te if y[i] == 1 and s[i] < t]
print(len(missed), '개 놓침')
for i in sorted(missed, key=lambda i: s[i])[:5]:
    print(i + 1, round(s[i], 4), X[i])